In [ ]:
import json
import os
from PIL import Image

def convert_split_to_coco(dataset_path, split_name, output_file):
    """Convert một split cụ thể sang COCO format"""
    annotations = {
        "images": [],
        "annotations": [],
        "categories": []
    }
    
    # Categories (giống nhau cho tất cả splits)
    class_names = ["apple", "tangerine", "pear", "watermelon", "durian", 
                   "lemon", "grape", "pineapple", "dragon fruit", "korean melon", "cantaloupe"]
    
    for i, name in enumerate(class_names):
        annotations["categories"].append({
            "id": i,
            "name": name,
            "supercategory": "fruit"
        })
    
    images_dir = os.path.join(dataset_path, split_name, "images")
    labels_dir = os.path.join(dataset_path, split_name, "labels")
    
    annotation_id = 1
    
    for img_file in os.listdir(images_dir):
        if not img_file.endswith(('.jpg', '.jpeg', '.png')):
            continue
            
        # Get image info
        img_path = os.path.join(images_dir, img_file)
        img = Image.open(img_path)
        img_width, img_height = img.size
        
        image_id = len(annotations["images"]) + 1
        annotations["images"].append({
            "id": image_id,
            "file_name": img_file,  # Chỉ tên file, không có path
            "width": img_width,
            "height": img_height
        })
        
        # Get annotations
        label_file = img_file.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
        label_path = os.path.join(labels_dir, label_file)
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id, x_center, y_center, width, height = map(float, parts)
                        
                        # Convert to absolute coordinates
                        x1 = (x_center - width/2) * img_width
                        y1 = (y_center - height/2) * img_height
                        bbox_width = width * img_width
                        bbox_height = height * img_height
                        
                        annotations["annotations"].append({
                            "id": annotation_id,
                            "image_id": image_id,
                            "category_id": int(class_id),
                            "bbox": [x1, y1, bbox_width, bbox_height],
                            "area": bbox_width * bbox_height,
                            "iscrowd": 0
                        })
                        annotation_id += 1
    
    # Save to JSON
    with open(output_file, 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f"Converted {split_name}: {len(annotations['images'])} images, {len(annotations['annotations'])} annotations")

# Convert tất cả splits
def convert_all_splits(dataset_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    splits = ["train", "validation", "test"]
    
    for split in splits:
        output_file = os.path.join(output_dir, f"{split}_annotations.json")
        convert_split_to_coco(dataset_path, split, output_file)

# Usage
convert_all_splits('./fruit_object_detection', './fruit_object_detection/annotations')

Converted train: 3076 images, 16146 annotations
Converted validation: 769 images, 4131 annotations
Converted test: 640 images, 2428 annotations
